### Data Cleaning and Modelling Transformation

In [28]:
import pandas as pd
import os

In [29]:
print(os.getcwd())

/Users/jack/Desktop/Fall 2026/CS4100/Final/CS4100-Final-Project/src/preprocessing


In [30]:
small_business_data = pd.read_csv("../../data/business_loans_2010_2025.csv")
small_business_data.columns

Index(['asofdate', 'program', 'l2locid', 'borrname', 'borrstreet', 'borrcity',
       'borrstate', 'borrzip', 'cdc_name', 'cdc_street', 'cdc_city',
       'cdc_state', 'cdc_zip', 'thirdpartylender_name',
       'thirdpartylender_city', 'thirdpartylender_state', 'thirdpartydollars',
       'grossapproval', 'approvaldate', 'approvalfiscalyear',
       'firstdisbursementdate', 'processingmethod', 'deliverymethod',
       'subprogram', 'terminmonths', 'naicscode', 'naicsdescription',
       'franchisecode', 'franchisename', 'projectcounty', 'projectstate',
       'sbadistrictoffice', 'congressionaldistrict', 'businesstype',
       'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate',
       'grosschargeoffamount', 'jobssupported', 'collateralind'],
      dtype='object')

In [31]:
small_business_data.loc[0]

asofdate                                                         12/31/2025
program                                                                 504
l2locid                                                            188355.0
borrname                             Hsu & Hsu Internal Medicine Associates
borrstreet                                   Intersection of U.S. Hwy 75 an
borrcity                                                              Allen
borrstate                                                                TX
borrzip                                                               75013
cdc_name                                     North Texas Certified Developm
cdc_street                                                  1700 Alma Drive
cdc_city                                                              Plano
cdc_state                                                                TX
cdc_zip                                                             75075.0
thirdpartyle

In [32]:
# Subset of data columns needed
filtered_sb_data = small_business_data[['asofdate', 'program', 'borrname', 'borrstate', 'borrzip', 
                                        'thirdpartydollars', 'terminmonths',
                                        'grossapproval', 'approvaldate', 'approvalfiscalyear', 
                                        'firstdisbursementdate', 'processingmethod', 'naicscode', 'collateralind',
                                        'businesstype', 'loanstatus', 'jobssupported',
                                         'grosschargeoffamount']].copy()

In [33]:
filtered_sb_data["approvalyear"] = pd.to_datetime(filtered_sb_data["approvaldate"]).dt.year
filtered_sb_data["approvalyear"] = (filtered_sb_data["approvalyear"] - 2010) / 15
filtered_sb_data.drop("approvaldate", axis=1, inplace=True)

In [51]:
filtered_sb_data

,asofdate,program,borrname,borrzip,thirdpartydollars,terminmonths,grossapproval,approvalfiscalyear,firstdisbursementdate,loanstatus,...,processingmethod_Premier Certified Lenders Program,businesstype_CORPORATION,businesstype_INDIVIDUAL,businesstype_PARTNERSHIP,collateralind_N,collateralind_Y,naicscode_0.0,naicscode_541690.0,naicscode_812990.0,leverageratio
1,12/31/2025,504,Ikona Inc.,2026,165000.0,240,137000,2010,11/17/2010,PIF,...,False,True,False,False,False,True,False,True,False,0.830303
2,12/31/2025,504,Stylz by Steffanie,2189,81250.0,240,68000,2010,7/14/2010,PIF,...,False,False,True,False,False,True,False,False,True,0.836923
3,12/31/2025,504,Jay Manufacturing Oshkosh Inc.,54904,1200000.0,120,969000,2010,3/16/2011,CHGOFF,...,False,True,False,False,False,True,True,False,False,0.807500
4,12/31/2025,504,CDI Thermoforming,791,1250000.0,240,1007000,2010,2/15/2012,PIF,...,False,True,False,False,False,True,True,False,False,0.805600
7,12/31/2025,504,American Steel Inc.,59101,1818898.0,240,1468000,2010,8/11/2010,PIF,...,False,True,False,False,False,True,True,False,False,0.807082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19990,12/31/2025,504,Fitness 2020,30253,1196500.0,300,865000,2024,4/17/2024,PIF,...,False,True,False,False,False,True,True,False,False,0.722942
20156,12/31/2025,504,Timberline Enterprises LLC,2341,687500.0,300,560000,2024,10/16/2024,PIF,...,False,True,False,False,False,True,True,False,False,0.814545
20208,12/31/2025,504,Kalaveras PD Inc,92860,1343300.0,300,964000,2024,3/13/2024,PIF,...,False,True,False,False,False,True,True,False,False,0.717636
20320,12/31/2025,504,"SUGARTREE, INC.",76462,629888.0,300,454000,2024,1/17/2024,PIF,...,False,True,False,False,False,True,True,False,False,0.720763


In [35]:
# Drop all nan business types
filtered_sb_data = filtered_sb_data.dropna(subset=['businesstype'])
filtered_sb_data['businesstype'].count()

np.int64(23122)

Only can look at loans that have meainingful default (CHGOFF) or fully paid (PIF) labels for default prediction model.

In [36]:
print(filtered_sb_data.shape)
filtered_sb_data = filtered_sb_data[filtered_sb_data["loanstatus"].isin(["PIF", "CHGOFF"])]
print(filtered_sb_data.shape)

(23122, 18)
(6842, 18)


In [52]:
filtered_sb_data['grosschargeoffamount'].unique()

array([      0,  275853,  270002,  878990,  398212,  135025,   66688,
        265917,  905484,  903210,  429853,  212608,   50649,  105870,
        254041,  116622,  296382,  212866,  344213,  378458, 1057417,
        508655,  168224,  257659,   94381,  618537,  220174,  500026,
         87854,  286046, 1202627,   39656, 1697130,   64158,   47038,
        435232, 1182967,  428151,  922540,  120449, 1300964,   57461,
        434294,  284536,  136548,  111145, 1162906,  544995,  581811,
        125881,  308306,  630022,  118715,  309959, 1904984,   77264,
        330061,  167536,   14336,  202636,  295971, 1891129,  393568,
        228384,  726480, 1095400,   39581,  579589,  386601,  447350,
        106642,  926159,  263438,  318903, 1736648,  546013,  758159,
         56545,   74643, 1384752,   37676,  301495,  100822,  123217,
         61190,  404410,  725859,  250926,  123894,   94705,  547938,
        182601,  698482,  315166,  794356,  781926,  455892,  186352,
        549966,   40

In [38]:
print(filtered_sb_data.isnull().sum())

asofdate                   0
program                    0
borrname                   0
borrstate                  0
borrzip                    0
thirdpartydollars        306
terminmonths               0
grossapproval              0
approvalfiscalyear         0
firstdisbursementdate      0
processingmethod           0
naicscode                 71
collateralind              0
businesstype               0
loanstatus                 0
jobssupported              0
grosschargeoffamount       0
approvalyear               0
dtype: int64


In [39]:
print(filtered_sb_data["businesstype"].unique().shape[0])

3


Filter NAICS code to first tow digits which specify industry for one hot encoding

In [40]:
filtered_sb_data["naicscode"] = filtered_sb_data["naicscode"][:2].astype(int)
filtered_sb_data["naicscode"] = filtered_sb_data["naicscode"].fillna(0)

In [41]:
encode_cols = [
    "borrstate", "processingmethod", "businesstype", 
    "collateralind", "naicscode"
]

In [42]:
filtered_sb_data = pd.get_dummies(filtered_sb_data, columns=encode_cols)

In [43]:
filtered_sb_data["thirdpartydollars"] = filtered_sb_data["thirdpartydollars"].fillna(filtered_sb_data["thirdpartydollars"].median())

Average SBA loan term

In [44]:
print(filtered_sb_data["terminmonths"].mean() / 12)

19.86407483192049


Derive leverage ratio col

In [45]:
filtered_sb_data["leverageratio"] = filtered_sb_data["grossapproval"] / filtered_sb_data["thirdpartydollars"]

In [46]:
filtered_sb_data.groupby("loanstatus").count()

,asofdate,program,borrname,borrzip,thirdpartydollars,terminmonths,grossapproval,approvalfiscalyear,firstdisbursementdate,jobssupported,...,processingmethod_Premier Certified Lenders Program,businesstype_CORPORATION,businesstype_INDIVIDUAL,businesstype_PARTNERSHIP,collateralind_N,collateralind_Y,naicscode_0.0,naicscode_541690.0,naicscode_812990.0,leverageratio
loanstatus,,,,,,,,,,,,,,,,,,,,,
CHGOFF,163,163,163,163,163,163,163,163,163,163,...,163,163,163,163,163,163,163,163,163,163
PIF,6679,6679,6679,6679,6679,6679,6679,6679,6679,6679,...,6679,6679,6679,6679,6679,6679,6679,6679,6679,6679


Extremely high class imbalance which we will have to account for during training.

In [63]:
filtered_sb_data["terminmonths"] = filtered_sb_data["terminmonths"].map({120:0, 240:1, 300:2}) / 2


In [65]:
filtered_sb_data.shape
    

(6842, 82)

Save data and examine output

In [48]:
filtered_sb_data.to_csv("../../data/cleaned_business_loans_2010_2025.csv", index=False)
# for element in zip(filtered_sb_data.iloc[0], filtered_sb_data.columns):
#     print(element)